# BACE general local-BFS — fixed-Q, variable-T scan

This notebook holds Q fixed at `8, 12, 16, 20` and scans `T=1,...,30`. It is the clean memory-length comparison: token width and reservoir/readout width remain fixed within each Q panel while only the exposed BFS event prefix grows.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'core' / 'linear_readout.py').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not locate 2_linear_readout project root')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from core.linear_readout import (
    DEFAULT_SEEDS, ExperimentConfig, prepare_fixed_q_layout,
    run_scan, collect_scan_results, plot_fixed_q_scan,
)
print('project:', PROJECT_ROOT)
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())

## Fixed protocol

All methods share the same task-independent BFS stream. QRC and CRC share `tanh` injection preprocessing, Q, three reservoir layers, and the final `Q(Q+1)/2` feature dimension. Raw directly flattens the original `T x 3Q` tokens without reservoir preprocessing.

In [ ]:
SCAN_ROOT = PROJECT_ROOT / 'general' / 'BACE' / '02_fixed_Q_T_scan'
DATA_DIR = PROJECT_ROOT / 'general' / 'BACE' / 'data'
SETTINGS = prepare_fixed_q_layout(SCAN_ROOT)
CFG = ExperimentConfig(
    dataset='BACE', data_dir=str(DATA_DIR), scan_root=str(SCAN_ROOT),
    epochs=500, seeds=DEFAULT_SEEDS, W=0.4, g=0.84, layers=3,
    qrc_devices=('cuda:0', 'cuda:1'), run_qrc=True, force=True,
)
print(CFG)
print('settings:', len(SETTINGS), SETTINGS[0], '...', SETTINGS[-1])

## Train and evaluate

Every setting has its own folder. Every seed has its own split, QRC/CRC features, linear checkpoint, 500-epoch history, and test predictions. Interrupted scans resume from completed artifacts.

In [ ]:
results = run_scan(CFG, SETTINGS)
results.tail(12)

## Result visualization

Each panel fixes Q. Lines show ten-seed means and bands show one standard deviation. Exports include editable SVG/PDF and source CSV.

In [ ]:
summary = collect_scan_results(SCAN_ROOT)
display(summary)
fig = plot_fixed_q_scan(SCAN_ROOT, 'BACE')
fig